# LLM–Brain Alignment — Surprisal vs Structure (Pereira2018)

Colab driver. Clones the repo, installs deps, runs the 5-stage pipeline, and plots
layer-depth curves + variance partitioning. See `README.md`, `LOG.md`, `TRACKER.md`.

**Core question:** how much LLM–brain alignment on Pereira2018 is *uniquely* explained by
hidden states beyond surprisal, and how does that depend on size / layer / instruction tuning.

## 1. Setup — clone + install

In [ ]:
# In Colab: clone the repo and install.
import os
REPO = 'LLMBrainAllignment'
if not os.path.exists(REPO):
    !git clone https://github.com/Sudarssan-N/LLMBrainAllignment.git
%cd {REPO}
!pip -q install -e .[models,plot]

In [ ]:
# Install the brain-score language harness for REAL Pereira2018 (needed for Section 3+).
# Heavy install (a few minutes). Skips if already present. Restart runtime if prompted.
import importlib.util
if importlib.util.find_spec('brainscore_language') is None:
    !pip -q install git+https://github.com/brain-score/language
else:
    print('brainscore_language already installed')

## 2. Smoke test (synthetic, no models)
Confirms the full chain works before pulling weights or data.

In [ ]:
!python -m pytest tests/ -q
for stage in ['01_extract_activations','02_compute_surprisal','03_fit_encoding',
              '04_variance_partitioning','05_rsa']:
    !python scripts/{stage}.py --synthetic

## 3. Real run — one model
**Run Section 1 install + the brain-score harness cell above first** (the harness provides
real Pereira2018; without it the loader raises by design rather than using synthetic data).

Set `MODEL` to a key from `config/default.yaml` (gpt2, pythia-160m, pythia-410m, opt-125m, ...).

In [ ]:
MODEL = 'gpt2'
!python scripts/01_extract_activations.py --model {MODEL}
!python scripts/02_compute_surprisal.py  --model {MODEL}
!python scripts/03_fit_encoding.py       --model {MODEL}
!python scripts/04_variance_partitioning.py --model {MODEL}
!python scripts/05_rsa.py                --model {MODEL}

## 4. Plots — layer-depth predictivity & variance partitioning

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

MODEL = 'gpt2'
d = Path('data/derived')/MODEL
enc = json.load(open(d/'encoding.json'))
vp  = json.load(open(d/'varpart.json'))

fig, ax = plt.subplots(1, 2, figsize=(12,4))
layers = sorted(int(l) for l in enc['per_layer'])
# prefer noise-ceiling-normalized predictivity when available
use_norm = enc.get('has_noise_ceiling') and enc['per_layer'][str(layers[0])].get('normalized_r') is not None
ykey = 'normalized_r' if use_norm else 'mean_r'
ax[0].plot(layers, [enc['per_layer'][str(l)][ykey] for l in layers], 'o-')
ax[0].set(title=f'{MODEL}: layer-depth predictivity ({ykey})', xlabel='layer',
          ylabel=ykey + (' (r / noise ceiling)' if use_norm else ' (raw r)'))

vl = sorted(int(l) for l in vp['per_layer'])
uh = [vp['per_layer'][str(l)]['mean_unique_hidden'] for l in vl]
us = [vp['per_layer'][str(l)]['mean_unique_surprisal'] for l in vl]
sh = [vp['per_layer'][str(l)]['mean_shared'] for l in vl]
ax[1].stackplot(vl, uh, sh, us, labels=['unique hidden','shared','unique surprisal'])
ax[1].set(title=f'{MODEL}: variance partitioning', xlabel='layer', ylabel='R²')
ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()
print('best layer:', enc['best_layer'], '| normalized_r:', enc.get('best_normalized_r'))

## 5. Model-family sweep
Run several models, then compare best-layer unique-hidden across size / instruction tuning.
Record each run in `TRACKER.md`.

In [ ]:
MODELS = ['gpt2','pythia-160m','pythia-410m','opt-125m']
for m in MODELS:
    for stage in ['01_extract_activations','02_compute_surprisal',
                  '03_fit_encoding','04_variance_partitioning','05_rsa']:
        !python scripts/{stage}.py --model {m}